In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = (
            SparkSession
                .builder
                .master("local[*]")
                .appName("notebook_teste")
                .config(
                    "spark.jars.packages",
                    "org.apache.hadoop:hadoop-aws:3.3.4,"
                    "com.amazonaws:aws-java-sdk-bundle:1.12.261,"
                    "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.1,"
                    "org.scala-lang:scala-library:2.13.12")
                # Configuração do Catálogo Iceberg
                .config("spark.sql.catalog.icebergtb", "org.apache.iceberg.spark.SparkCatalog")
                .config("spark.driver.extraJavaOptions", "-Dlog4j.configuration=log4j.properties") 
                .config("spark.log.level", "ERROR") 
                .config("spark.eventLog.enabled", "false") 
                # .config("spark.sql.catalog.icebergtb.catalog-impl", "org.apache.iceberg.hadoop.HadoopCatalog")
                .config("spark.sql.catalog.icebergtb.type", "hadoop") 
                .config("spark.sql.catalog.icebergtb.warehouse", "s3a://iceberg/")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.endpoint", "http://172.19.192.1:9000")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.access.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.secret.key", "minioadmin")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.path.style.access", "true")
                .config("spark.sql.catalog.icebergtb.hadoop.fs.s3a.connection.ssl.enabled", "false")
                .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
                .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") 
            .getOrCreate()
)

spark

In [ ]:
dataframe = (spark.sql(""" select 
                            row_number() over(order by businessentityid) as supplierkey,
                            businessentityid as supplierid,
                            productid as productseller_id,
                            *
                            from icebergtb.advworks_rdb.raw_dimsupplier""")
             .drop(col("businessentityid"))
             .drop(col("productid"))
)

In [ ]:
dataframe = (dataframe
                 .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bln\b","Lane"))
                .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\brd\b","Road"))
                .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bDr\b","Drive"))
                .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bCt\b","Center"))
                .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bst\b","Street"))
                .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bblvd\b", "Boulevard"))
                .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bave\b","Avenue"))
                .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bpk\b","Parkway"))
                .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bterr\b", "Terrace"))
                .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bpl\b","Plaza"))
                .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bmt\b","Mountain"))
                .withColumn("addressline1", regexp_replace("addressline1", r"(?i)\bn.\b","North"))
            )

In [ ]:
dataframe = (dataframe
             .withColumn("suppliername", regexp_replace("name", r"(?i)\bcorp\b", "Corporation"))
             .withColumn("suppliername", regexp_replace("name", r"(?i)\bco\b", "Company"))
             .withColumn("suppliername", regexp_replace("name", r"(?i)\binc\b", "Incorporation"))
             
             
             
)

In [ ]:
dataframe = (dataframe 
             .withColumn("addressnumber", regexp_extract("addressline1", r"(\d+ )", 1))
             .withColumn("supplieraddress", regexp_replace("addressline1", r"[^a-zA-Z ]", ""))
             .withColumn("addressline", coalesce(col("addressline2"), lit("Not Available")))
             .drop(col("addressline1"))
             .drop(col("addressline2"))
             
             
)

In [ ]:
dataframe = (dataframe
                .selectExpr('*',
                            
                    'coalesce(onorderqty, 0) as orderqty',
                    
                    """
                    case 
                        when creditrating = 1 then 'Premier'
                        when creditrating between 2 and 3 then 'Super class'
                        when creditrating = 4 then 'Normal'
                        when creditrating = 5 then 'Attention'
                    else 'No Classification'
                    end as ratingclass
                    """,
                    
                    """
                    case
		                when preferredvendorstatus = 1 then 'Yes'
		                else  'No'
	                end as ispreferred
                    """
                ))

In [ ]:
dataframe.select(
    "supplierkey",          "supplierid",
    "productseller_id",     "accountnumber",
    "name",                 "contacttype",
    "firstname",            "lastname",
    "phonenumber",          "phonenumbertype",
    "creditrating",         "preferredvendorstatus",
    "averageleadtime",      "standardprice",
    "lastreceiptcost",      "lastreceiptdate",
    "minorderqty",          "maxorderqty",
    "onorderqty",           "unitmeasurecode",
    "stateprovincename",    "countryregionname",
    "city",                 "postalcode",
    "inserted_date",        "modified_date",
    "eltversion",           "suppliername",
    "addressnumber",        "supplieraddress",
    "addressline",          "orderqty",
    "ratingclass",          "ispreferred")